In [ ]:
import os
import shutil
import gc
import json
import time
import copy
import toml
import torch
import joblib
import datetime
import argparse
import subprocess

import pandas as pd
import numpy as np

from config_io import Config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from GPUtil import showUtilization as gpu_usage
# from config import configs # run once to ensure the latest configs are loaded
# print("Finish loading configs")
# from config_small_scale import NETGPT_BASE_FOLDER
from torch.profiler import profile, record_function, ProfilerActivity
import netshare.ray as ray
from netshare import Generator

In [ ]:
work_folder = "./netshare_work_folder"
if os.path.exists(work_folder):
    shutil.rmtree(work_folder)
os.makedirs(work_folder, exist_ok=True)

ray.config.enabled = False
ray.init(address="auto")
generator = Generator(config="netshare_config.json")
generator.train_and_generate(work_folder=work_folder)
print(f"Generated file list: {generator._pre_post_processor.best_syndf_filename_list}")
print(f"Best Gen trace: {generator._pre_post_processor.best_syndf_filename_list[0]}")
syn_df = pd.read_csv(generator._pre_post_processor.best_syndf_filename_list[0])
syn_df.to_csv("../data/generated_netshare_caida_10k.csv", index=False)
print("Saved generated data to ../data/generated_netshare_caida_10k.csv")
ray.shutdown()